In [1]:
%pip install -q langgraph

Note: you may need to restart the kernel to use updated packages.


# Resilient Legal Document Processing

Task 1: define a shared LangGraph state that supports contract analysis, resilience tracking, and compliance auditing.

In [2]:
from datetime import datetime, timezone
from typing import Any, Literal, NotRequired, TypedDict

from langgraph.graph import END, START, StateGraph


In [3]:
DocumentComplexity = Literal["simple", "standard", "complex"]
UrgencyLevel = Literal["standard", "rush"]
SystemHealth = Literal["healthy", "degraded", "offline"]
ProcessingStatus = Literal["pending", "processing", "completed", "partial", "manual_review"]


class ProcessingHistory(TypedDict):
    document_type: str
    prior_failures: int
    last_failure_reason: NotRequired[str]
    last_successful_route: NotRequired[str]


class ProcessingStep(TypedDict):
    step: str
    status: Literal["success", "failure", "fallback"]
    timestamp: str
    details: str


class ErrorLog(TypedDict):
    node: str
    error_type: str
    message: str
    retry_count: int
    fallback_used: bool
    timestamp: str


class AuditEntry(TypedDict):
    event: str
    actor: str
    timestamp: str
    data_sources: list[str]


class LegalDocumentState(TypedDict):
    """Shared state for a resilient legal document processing workflow."""

    document_id: str
    document_content: str
    urgency: UrgencyLevel
    system_health: SystemHealth
    processing_history: ProcessingHistory

    status: NotRequired[ProcessingStatus]
    complexity: NotRequired[DocumentComplexity]
    extracted_text: NotRequired[str]
    extracted_fields: NotRequired[dict[str, Any]]
    validation_results: NotRequired[dict[str, Any]]
    summary: NotRequired[str]

    processing_steps: NotRequired[list[ProcessingStep]]
    retry_counts: NotRequired[dict[str, int]]
    fallback_usage: NotRequired[list[str]]
    error_logs: NotRequired[list[ErrorLog]]

    audit_trail: NotRequired[list[AuditEntry]]
    data_sources_used: NotRequired[list[str]]
    compliance_notes: NotRequired[list[str]]


## State Schema

| Area | Fields | Purpose |
| --- | --- | --- |
| Business data | `document_id`, `document_content`, `extracted_text`, `extracted_fields`, `validation_results`, `summary` | Stores the contract and the useful outputs produced by the workflow |
| Execution tracking | `status`, `complexity`, `processing_steps` | Records which steps ran, when they ran, and whether each step succeeded |
| Error handling | `retry_counts`, `fallback_usage`, `error_logs` | Captures retries, failures, and fallback mechanisms for debugging |
| Compliance | `audit_trail`, `data_sources_used`, `compliance_notes` | Maintains an auditable trail of processing decisions and data sources |

## Task 2: Error Handling Plan

| Failure scenario | Detection method | Fallback strategy | State updates |
| --- | --- | --- | --- |
| External API failure | OCR or legal database is marked unavailable by `system_health`, or the document contains simulated service outage markers | Use embedded text extraction for OCR and an internal legal clause library for validation | Add `ErrorLog`, increment `retry_counts`, append `fallback_usage`, add source to `data_sources_used` |
| Data quality issue | Empty content, unreadable content, or corruption markers such as `CORRUPTED` / `UNREADABLE` | Stop automated processing and send the document to manual review with partial notes | Set `status` to `manual_review`, add compliance note, write error and audit records |
| Processing timeout | Complex contracts, high page counts, or explicit timeout markers | Switch from full-document processing to chunked processing | Record timeout error, add `chunked_processing` fallback, continue with partial but usable extraction |

In [ ]:
def now() -> str:
    return datetime.now(timezone.utc).isoformat()


def add_step(state: LegalDocumentState, step: str, status: str, details: str) -> list[ProcessingStep]:
    return [
        *state.get("processing_steps", []),
        {"step": step, "status": status, "timestamp": now(), "details": details},
    ]


def add_error(
    state: LegalDocumentState,
    node: str,
    error_type: str,
    message: str,
    retry_count: int = 0,
    fallback_used: bool = False,
) -> list[ErrorLog]:
    return [
        *state.get("error_logs", []),
        {
            "node": node,
            "error_type": error_type,
            "message": message,
            "retry_count": retry_count,
            "fallback_used": fallback_used,
            "timestamp": now(),
        },
    ]


def add_audit(state: LegalDocumentState, event: str, data_sources: list[str]) -> list[AuditEntry]:
    return [
        *state.get("audit_trail", []),
        {"event": event, "actor": "langgraph_workflow", "timestamp": now(), "data_sources": data_sources},
    ]


## Workflow Nodes

| Node | Processing | Resilience behavior |
| --- | --- | --- |
| `assess_document` | Estimates complexity from length, parties, clauses, and history | Initializes status, audit trail, retry counters, and data sources |
| `fast_track_extract` | Extracts text and fields for simple urgent contracts | Uses lightweight extraction to reduce delay |
| `standard_extract` | Runs normal OCR-style extraction | Detects API failure, corruption, and timeout markers |
| `chunked_processing` | Processes complex documents in smaller sections | Handles timeout risk and produces partial but usable output |
| `validate_clauses` | Validates key legal clauses | Falls back to an internal clause library if the external database is unavailable |
| `generate_summary` | Produces a concise contract summary | Marks full or partial completion |
| `manual_review` | Sends failed documents to human review | Preserves logs and compliance notes |

In [ ]:
def assess_document(state: LegalDocumentState) -> LegalDocumentState:
    content = state["document_content"]
    lower_content = content.lower()
    history = state.get("processing_history", {})

    if len(content) > 1200 or "multi-party" in lower_content or "master services agreement" in lower_content:
        complexity: DocumentComplexity = "complex"
    elif len(content) > 400 or "indemnity" in lower_content or "liability" in lower_content:
        complexity = "standard"
    else:
        complexity = "simple"

    return {
        **state,
        "status": "processing",
        "complexity": complexity,
        "processing_steps": add_step(state, "assess_document", "success", f"Document classified as {complexity}."),
        "retry_counts": {"ocr": 0, "legal_database": 0, **state.get("retry_counts", {})},
        "fallback_usage": state.get("fallback_usage", []),
        "error_logs": state.get("error_logs", []),
        "audit_trail": add_audit(state, "document_assessed", ["document_metadata", "processing_history"]),
        "data_sources_used": [*state.get("data_sources_used", []), "document_metadata"],
        "compliance_notes": [
            *state.get("compliance_notes", []),
            f"Prior failures for this document type: {history.get('prior_failures', 0)}",
        ],
    }


def content_has_quality_issue(content: str) -> bool:
    normalized = content.strip().lower()
    return not normalized or "corrupted" in normalized or "unreadable" in normalized


def extract_fields(text: str) -> dict[str, Any]:
    lower_text = text.lower()
    return {
        "parties_detected": "party" in lower_text or "between" in lower_text,
        "effective_date_detected": "effective date" in lower_text,
        "termination_clause_detected": "termination" in lower_text,
        "liability_clause_detected": "liability" in lower_text,
        "confidentiality_clause_detected": "confidential" in lower_text,
    }


def fast_track_extract(state: LegalDocumentState) -> LegalDocumentState:
    content = state["document_content"]
    if content_has_quality_issue(content):
        return {
            **state,
            "status": "manual_review",
            "error_logs": add_error(state, "fast_track_extract", "data_quality", "Document content is empty or unreadable."),
            "processing_steps": add_step(state, "fast_track_extract", "failure", "Quality issue found during fast-track extraction."),
            "compliance_notes": [*state.get("compliance_notes", []), "Manual review required because source text failed quality checks."],
        }

    return {
        **state,
        "extracted_text": content,
        "extracted_fields": extract_fields(content),
        "processing_steps": add_step(state, "fast_track_extract", "success", "Used lightweight extraction for rush/simple document."),
        "audit_trail": add_audit(state, "fast_track_extraction_completed", ["embedded_document_text"]),
        "data_sources_used": [*state.get("data_sources_used", []), "embedded_document_text"],
    }


def standard_extract(state: LegalDocumentState) -> LegalDocumentState:
    content = state["document_content"]
    lower_content = content.lower()

    if content_has_quality_issue(content):
        return {
            **state,
            "status": "manual_review",
            "error_logs": add_error(state, "standard_extract", "data_quality", "Document is corrupted or unreadable."),
            "processing_steps": add_step(state, "standard_extract", "failure", "Automated extraction stopped because content is unreadable."),
            "compliance_notes": [*state.get("compliance_notes", []), "Source document requires human inspection before legal conclusions are made."],
        }

    if state["system_health"] in ["degraded", "offline"] or "ocr outage" in lower_content:
        retry_counts = {**state.get("retry_counts", {})}
        retry_counts["ocr"] = retry_counts.get("ocr", 0) + 1
        return {
            **state,
            "extracted_text": content,
            "extracted_fields": extract_fields(content),
            "retry_counts": retry_counts,
            "fallback_usage": [*state.get("fallback_usage", []), "embedded_text_extraction"],
            "error_logs": add_error(state, "standard_extract", "external_api_failure", "OCR service unavailable.", retry_counts["ocr"], True),
            "processing_steps": add_step(state, "standard_extract", "fallback", "OCR failed; used embedded text extraction."),
            "audit_trail": add_audit(state, "ocr_fallback_used", ["embedded_document_text"]),
            "data_sources_used": [*state.get("data_sources_used", []), "embedded_document_text"],
        }

    if state.get("complexity") == "complex" or "timeout" in lower_content:
        return {
            **state,
            "error_logs": add_error(state, "standard_extract", "processing_timeout", "Full-document extraction exceeded the time budget.", 0, True),
            "fallback_usage": [*state.get("fallback_usage", []), "chunked_processing"],
            "processing_steps": add_step(state, "standard_extract", "fallback", "Timeout risk detected; rerouting to chunked processing."),
        }

    return {
        **state,
        "extracted_text": content,
        "extracted_fields": extract_fields(content),
        "processing_steps": add_step(state, "standard_extract", "success", "OCR extraction completed."),
        "audit_trail": add_audit(state, "ocr_extraction_completed", ["ocr_service"]),
        "data_sources_used": [*state.get("data_sources_used", []), "ocr_service"],
    }


def chunked_processing(state: LegalDocumentState) -> LegalDocumentState:
    content = state["document_content"]
    if content_has_quality_issue(content):
        return {
            **state,
            "status": "manual_review",
            "error_logs": add_error(state, "chunked_processing", "data_quality", "Chunked processing cannot read source content."),
            "processing_steps": add_step(state, "chunked_processing", "failure", "Unreadable chunks require manual review."),
        }

    existing_timeout_errors = [error for error in state.get("error_logs", []) if error["error_type"] == "processing_timeout"]
    error_logs = state.get("error_logs", [])
    if not existing_timeout_errors:
        error_logs = add_error(
            state,
            "chunked_processing",
            "processing_timeout",
            "Chunked processing selected to keep the document within the processing time budget.",
            0,
            True,
        )

    chunks = [content[i : i + 300] for i in range(0, len(content), 300)]
    partial_text = "\n".join(chunks[:4])
    return {
        **state,
        "extracted_text": partial_text,
        "extracted_fields": extract_fields(partial_text),
        "fallback_usage": [*state.get("fallback_usage", []), "chunked_processing"],
        "error_logs": error_logs,
        "processing_steps": add_step(state, "chunked_processing", "fallback", f"Processed {len(chunks)} chunks with partial extraction."),
        "audit_trail": add_audit(state, "chunked_processing_completed", ["document_chunks"]),
        "data_sources_used": [*state.get("data_sources_used", []), "document_chunks"],
        "compliance_notes": [*state.get("compliance_notes", []), "Summary may be partial because chunked processing was used."],
    }


In [ ]:
def validate_clauses(state: LegalDocumentState) -> LegalDocumentState:
    fields = state.get("extracted_fields", {})
    missing_clauses = [
        clause
        for clause, detected in {
            "termination": fields.get("termination_clause_detected", False),
            "liability": fields.get("liability_clause_detected", False),
            "confidentiality": fields.get("confidentiality_clause_detected", False),
        }.items()
        if not detected
    ]

    legal_database_available = state["system_health"] == "healthy" and "legal db outage" not in state["document_content"].lower()
    source = "external_legal_database" if legal_database_available else "internal_clause_library"
    fallback_usage = state.get("fallback_usage", [])
    error_logs = state.get("error_logs", [])
    retry_counts = {**state.get("retry_counts", {})}
    step_status = "success"
    details = "Validated clauses against external legal database."

    if not legal_database_available:
        retry_counts["legal_database"] = retry_counts.get("legal_database", 0) + 1
        fallback_usage = [*fallback_usage, "internal_clause_library"]
        error_logs = add_error(
            state,
            "validate_clauses",
            "external_api_failure",
            "Legal database unavailable.",
            retry_counts["legal_database"],
            True,
        )
        step_status = "fallback"
        details = "Legal database unavailable; used internal clause library."

    validation_results = {
        "source": source,
        "missing_clauses": missing_clauses,
        "risk_level": "high" if len(missing_clauses) >= 2 else "medium" if missing_clauses else "low",
        "requires_attorney_review": bool(missing_clauses) or state.get("complexity") == "complex",
    }

    return {
        **state,
        "validation_results": validation_results,
        "retry_counts": retry_counts,
        "fallback_usage": fallback_usage,
        "error_logs": error_logs,
        "processing_steps": add_step(state, "validate_clauses", step_status, details),
        "audit_trail": add_audit(state, "clauses_validated", [source]),
        "data_sources_used": [*state.get("data_sources_used", []), source],
    }


def generate_summary(state: LegalDocumentState) -> LegalDocumentState:
    fields = state.get("extracted_fields", {})
    validation = state.get("validation_results", {})
    fallback_count = len(state.get("fallback_usage", []))

    summary = (
        f"Document {state['document_id']} processed as a {state.get('complexity')} contract. "
        f"Parties detected: {fields.get('parties_detected', False)}. "
        f"Risk level: {validation.get('risk_level', 'unknown')}. "
        f"Missing clauses: {validation.get('missing_clauses', [])}. "
        f"Fallbacks used: {fallback_count}."
    )

    status: ProcessingStatus = "partial" if fallback_count else "completed"
    return {
        **state,
        "status": status,
        "summary": summary,
        "processing_steps": add_step(state, "generate_summary", "success", f"Generated {status} summary."),
        "audit_trail": add_audit(state, "summary_generated", ["workflow_outputs"]),
    }


def manual_review(state: LegalDocumentState) -> LegalDocumentState:
    error_logs = state.get("error_logs", [])
    if content_has_quality_issue(state["document_content"]) and not error_logs:
        error_logs = add_error(state, "manual_review", "data_quality", "Document content is empty, corrupted, or unreadable.")

    return {
        **state,
        "status": "manual_review",
        "summary": f"Document {state['document_id']} requires manual legal operations review before automated conclusions are used.",
        "error_logs": error_logs,
        "processing_steps": add_step(state, "manual_review", "success", "Queued document for human review."),
        "audit_trail": add_audit(state, "manual_review_queued", ["workflow_outputs"]),
        "compliance_notes": [*state.get("compliance_notes", []), "Human review preserves reliability when automation cannot verify the document."],
    }


## Task 3: Conditional Routing Logic

Decision tree:

1. `START -> assess_document`
2. If content is corrupted or unreadable, route to `manual_review`.
3. If the document is simple, urgent, and system health is healthy, route to `fast_track_extract`.
4. If the document is complex, has prior failures, or the system is degraded, route to `chunked_processing`.
5. Otherwise, route to `standard_extract`.
6. If extraction records a manual-review status, route to `manual_review`.
7. If extraction records a timeout fallback and has no extracted text yet, route to `chunked_processing`.
8. All usable extractions route to `validate_clauses`.
9. High-risk documents with multiple missing clauses route to `manual_review`; all others route to `generate_summary`.

Simple flowchart:

`START -> assess_document -> initial route -> extract/chunk/manual_review -> validate_clauses -> summary or manual_review -> END`

In [ ]:
def choose_initial_path(state: LegalDocumentState) -> Literal["fast_track_extract", "standard_extract", "chunked_processing", "manual_review"]:
    history = state.get("processing_history", {})

    if content_has_quality_issue(state["document_content"]):
        return "manual_review"
    if state.get("complexity") == "simple" and state["urgency"] == "rush" and state["system_health"] == "healthy":
        return "fast_track_extract"
    if state.get("complexity") == "complex" or history.get("prior_failures", 0) >= 2 or state["system_health"] == "degraded":
        return "chunked_processing"
    return "standard_extract"


def choose_after_extraction(state: LegalDocumentState) -> Literal["validate_clauses", "chunked_processing", "manual_review"]:
    if state.get("status") == "manual_review":
        return "manual_review"
    if "chunked_processing" in state.get("fallback_usage", []) and "extracted_text" not in state:
        return "chunked_processing"
    return "validate_clauses"


def choose_completion_path(state: LegalDocumentState) -> Literal["generate_summary", "manual_review"]:
    validation = state.get("validation_results", {})
    if validation.get("risk_level") == "high" and validation.get("requires_attorney_review"):
        return "manual_review"
    return "generate_summary"


workflow = StateGraph(LegalDocumentState)

workflow.add_node("assess_document", assess_document)
workflow.add_node("fast_track_extract", fast_track_extract)
workflow.add_node("standard_extract", standard_extract)
workflow.add_node("chunked_processing", chunked_processing)
workflow.add_node("validate_clauses", validate_clauses)
workflow.add_node("generate_summary", generate_summary)
workflow.add_node("manual_review", manual_review)

workflow.add_edge(START, "assess_document")
workflow.add_conditional_edges(
    "assess_document",
    choose_initial_path,
    {
        "fast_track_extract": "fast_track_extract",
        "standard_extract": "standard_extract",
        "chunked_processing": "chunked_processing",
        "manual_review": "manual_review",
    },
)
workflow.add_conditional_edges(
    "fast_track_extract",
    choose_after_extraction,
    {"validate_clauses": "validate_clauses", "chunked_processing": "chunked_processing", "manual_review": "manual_review"},
)
workflow.add_conditional_edges(
    "standard_extract",
    choose_after_extraction,
    {"validate_clauses": "validate_clauses", "chunked_processing": "chunked_processing", "manual_review": "manual_review"},
)
workflow.add_edge("chunked_processing", "validate_clauses")
workflow.add_conditional_edges(
    "validate_clauses",
    choose_completion_path,
    {"generate_summary": "generate_summary", "manual_review": "manual_review"},
)
workflow.add_edge("generate_summary", END)
workflow.add_edge("manual_review", END)

legal_document_graph = workflow.compile()


## Invoke and Visualize the Graph

Display the graph structure, then test it with documents that exercise the normal path, API fallback path, timeout/chunking path, and manual-review path.

In [ ]:
from IPython.display import Image, Markdown, display

try:
    display(Image(legal_document_graph.get_graph().draw_mermaid_png()))
except Exception:
    mermaid_diagram = legal_document_graph.get_graph().draw_mermaid()
    display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))


In [ ]:
from pprint import pprint

sample_documents = [
    {
        "document_id": "CON-001",
        "document_content": "Agreement between Party A and Party B. Effective Date is today. Confidential information is protected. Termination requires 30 days notice. Liability is capped.",
        "urgency": "rush",
        "system_health": "healthy",
        "processing_history": {"document_type": "nda", "prior_failures": 0},
    },
    {
        "document_id": "CON-002",
        "document_content": "Service agreement between Party A and Party B. OCR outage. Effective Date included. Confidential obligations exist. Termination clause present.",
        "urgency": "standard",
        "system_health": "offline",
        "processing_history": {"document_type": "service_agreement", "prior_failures": 1},
    },
    {
        "document_id": "CON-003",
        "document_content": "Master Services Agreement multi-party contract. " * 40 + "Termination, liability, and confidential terms appear across exhibits.",
        "urgency": "standard",
        "system_health": "degraded",
        "processing_history": {"document_type": "msa", "prior_failures": 2},
    },
    {
        "document_id": "CON-004",
        "document_content": "CORRUPTED UNREADABLE SCAN",
        "urgency": "rush",
        "system_health": "healthy",
        "processing_history": {"document_type": "unknown", "prior_failures": 0},
    },
]

for document in sample_documents:
    result = legal_document_graph.invoke(document)
    print("Document:", document["document_id"])
    pprint(
        {
            "status": result.get("status"),
            "complexity": result.get("complexity"),
            "fallback_usage": result.get("fallback_usage"),
            "error_logs": result.get("error_logs"),
            "validation_results": result.get("validation_results"),
            "summary": result.get("summary"),
        }
    )
    print("-" * 100)


## Resilience Rationale

- The state schema keeps business outputs, execution history, errors, fallback use, and compliance audit records together so every decision is explainable after processing.
- External service failures do not stop the workflow; OCR falls back to embedded text extraction and legal validation falls back to an internal clause library.
- Data quality failures are treated differently from service failures because bad source material can create unreliable legal conclusions; those documents are routed to manual review.
- Timeout-prone contracts move to chunked processing so the firm still receives partial extraction and risk signals instead of losing the whole job.
- Conditional routing considers complexity, system health, prior failures, and urgency, which lets the workflow adapt to operational pressure while preserving auditability.

## Completion Criteria

- The state schema supports business outputs, operational resilience, and compliance auditing.
- The workflow handles external API failures, data quality issues, and timeout risk with realistic fallbacks.
- Conditional routing uses document complexity, system health, processing history, and urgency.
- The runnable examples demonstrate completed, partial, fallback, and manual-review outcomes.